# Two-Layer Neural Network from Scratch: Two Moons Binary Classification

This is the **exercise version** of the notebook, on the `exercise/two-layer-nn-en` branch. The Chinese version is on the `exercise/two-layer-nn-zh` branch, and the two notebooks match cell for cell.

Using only NumPy, you will build a `2 → 5 → 1` two-layer neural network by hand: Forward Propagation, Binary Cross-Entropy, Backpropagation, Numerical Gradient Checking, and Batch Gradient Descent. The core functions come with full signatures, docstrings, and TODO hints; the function bodies are yours to write. Data loading, formulas, API links, self-checks, and plotting code are already provided.

## How to use this notebook

1. Run `uv sync` in this directory, then pick `.venv/bin/python` as the kernel in VS Code / Jupyter.
2. Run from top to bottom. When you reach a `TODO` function, implement it, then run the self-check cell right after it.
3. After editing a function, re-run the cell that defines it. The kernel does not pick up editor changes on its own.
4. Calling a function you have not implemented raises `NotImplementedError`. That is expected when you start.
5. Keep function names, argument order, and return conventions unchanged. Later self-checks and plots depend on them.
6. When everything is done, run Restart Kernel → Run All to make sure nothing depends on stale variables.

**Rules**: build the model with NumPy only. scikit-learn is only for generating the data and making the train/test split. No PyTorch, TensorFlow, automatic differentiation, or ready-made optimizers.

## Roadmap

| Lab | Topic | Functions you implement | Plots provided for observation |
| --- | --- | --- | --- |
| 1 | Dataset preparation | `standardize_features` | Scatter plots of raw and standardized data |
| 2 | Parameter initialization | `initialize_parameters` | Histogram of He-initialized weights |
| 3 | Forward Propagation | `relu`, `relu_derivative`, `sigmoid`, `forward` | Activation functions and their derivatives |
| 4 | Loss and Backpropagation | `compute_loss`, `backward`, `gradient_check` | Bar chart of gradient-check relative errors |
| 5 | Training loop | `update_parameters`, `train` | Loss curve (linear / log scale) |
| 6 | Evaluation and visualization | `predict_proba`, `predict`, `accuracy` | Decision boundaries, hidden-unit activation maps, training evolution |
| 7 | Experiments | None (change settings and observe) | Learning-rate comparison, hidden-width comparison |

That is 14 functions. `plot_decision_boundary` is provided as an observation tool and is not part of the exercise. For extra practice, rewrite it yourself after finishing all the Labs.

## 1. Problem Definition

**Task**: given a 2-D point $(x_1, x_2)$, predict whether it belongs to Class 0 or Class 1.

| Item | Setting |
| --- | --- |
| Input features | 2 ($x_1$, $x_2$) |
| Output classes | 2 (the model outputs one probability $\hat y = P(y=1 \mid x)$) |
| Dataset size | 1,000 samples, `noise=0.2`, `random_state=42` |
| Training / Test split | 80% / 20%, i.e. 800 / 200, stratified by class (`stratify=y`) |

**Why Two Moons?** The two classes look like two interlocking crescents. No straight line can separate them cleanly, so a linear model such as Logistic Regression gets stuck around 86% training accuracy. A hidden layer with a non-linear activation such as ReLU lets the network learn a curved decision boundary. Lab 6 plots both decision boundaries side by side.

Because the data is only 2-D, every step can be drawn: the data distribution, the region each hidden neuron "sees", and how the boundary gradually bends during training.

## 2. Environment and API Quick Reference

The tables list only the **simplified usage** this exercise needs, not every parameter of each API. Click a name to open the official documentation.

### 2.1 Data (provided, just read it)

| API (official docs) | Key arguments / return value | Used in |
| --- | --- | --- |
| [`sklearn.datasets.make_moons`](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.make_moons.html) | `n_samples`, `noise`, `random_state` → `X (n, 2)`, `y (n,)` | Generate the data |
| [`sklearn.model_selection.train_test_split`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html) | `test_size`, `random_state`, `stratify` → `X_train, X_test, y_train, y_test` | Split the data |
| [`ndarray.reshape(-1, 1)`](https://numpy.org/doc/stable/reference/generated/numpy.ndarray.reshape.html) | `-1` means "infer this dimension" → `(m,)` becomes `(m, 1)` | Turn labels into column vectors |
| [`np.unique(a, return_counts=True)`](https://numpy.org/doc/stable/reference/generated/numpy.unique.html) | Returns the distinct values and how often each appears | Check that labels are only 0 and 1 |

### 2.2 Creating arrays and random numbers

| API (official docs) | Key arguments / return value | Used in |
| --- | --- | --- |
| [`np.random.seed(seed)`](https://numpy.org/doc/stable/reference/random/generated/numpy.random.seed.html) | Fixes the global random state so results are reproducible | `initialize_parameters` |
| [`np.random.randn(d0, d1)`](https://numpy.org/doc/stable/reference/random/generated/numpy.random.randn.html) | Samples from the standard normal $\mathcal N(0,1)$, returns shape `(d0, d1)` | `initialize_parameters` |
| [`np.sqrt(x)`](https://numpy.org/doc/stable/reference/generated/numpy.sqrt.html) | Square root | Scale factor in He initialization |
| [`np.zeros(shape)`](https://numpy.org/doc/stable/reference/generated/numpy.zeros.html) | `shape` is a tuple, e.g. `(1, 5)` | Bias initialization |
| [`np.zeros_like(a)`](https://numpy.org/doc/stable/reference/generated/numpy.zeros_like.html) | All-zero array with the same shape and dtype as `a` | `gradient_check` stores numerical gradients |

### 2.3 Element-wise operations

| API (official docs) | Key arguments / return value | Used in |
| --- | --- | --- |
| [`np.maximum(a, b)`](https://numpy.org/doc/stable/reference/generated/numpy.maximum.html) | Element-wise maximum, broadcasts: `np.maximum(0, Z)` | `relu` |
| Comparisons `Z > 0`, `P >= t` | Return a boolean array | `relu_derivative` (`>`), `predict` (`>=`) |
| [`ndarray.astype(dtype)`](https://numpy.org/doc/stable/reference/generated/numpy.ndarray.astype.html) | Converts type, e.g. boolean → `float` or `int` | `relu_derivative`, `predict` |
| [`np.exp(x)`](https://numpy.org/doc/stable/reference/generated/numpy.exp.html) | Element-wise $e^x$ | `sigmoid` |
| [`np.log(x)`](https://numpy.org/doc/stable/reference/generated/numpy.log.html) | Natural log $\ln x$ (not `log10`) | `compute_loss` |
| [`np.clip(a, a_min, a_max)`](https://numpy.org/doc/stable/reference/generated/numpy.clip.html) | Limits values to an interval, returns a new array | `compute_loss`, to avoid `log(0)` |

### 2.4 Matrix operations and reductions

| API (official docs) | Key arguments / return value | Used in |
| --- | --- | --- |
| [`A @ B` (`np.matmul`)](https://numpy.org/doc/stable/reference/generated/numpy.matmul.html) | Matrix product: `(m, n) @ (n, k)` → `(m, k)` | `forward`, `backward` |
| [`A.T`](https://numpy.org/doc/stable/reference/generated/numpy.ndarray.T.html) | Transpose: `(m, n)` → `(n, m)` | `backward` |
| [`np.sum(a, axis=0, keepdims=True)`](https://numpy.org/doc/stable/reference/generated/numpy.sum.html) | `axis=0` sums over samples; `keepdims=True` keeps the array 2-D, e.g. `(m, 5)` → `(1, 5)` | `backward`, for `db1` and `db2` |
| [`np.mean(a, axis=0, keepdims=True)`](https://numpy.org/doc/stable/reference/generated/numpy.mean.html) | Mean; without `axis` it averages every element | `standardize_features`, `compute_loss`, `accuracy` |
| [`np.std(a, axis=0, keepdims=True)`](https://numpy.org/doc/stable/reference/generated/numpy.std.html) | Standard deviation, `ddof=0` by default (population std) | `standardize_features` |
| [`np.linalg.norm(x)`](https://numpy.org/doc/stable/reference/generated/numpy.linalg.norm.html) | L2 norm $\lVert x\rVert_2$ by default; matrices are treated as flattened | `gradient_check` relative error |
| [Broadcasting](https://numpy.org/doc/stable/user/basics.broadcasting.html) | `(m, 5) + (1, 5)` adds the bias to every row | `forward` |

### 2.5 Copying, iteration, and the training loop

| API (official docs) | Key arguments / return value | Used in |
| --- | --- | --- |
| [`ndarray.copy()`](https://numpy.org/doc/stable/reference/generated/numpy.ndarray.copy.html) | Returns an independent copy; changing the copy leaves the original alone | `gradient_check` |
| [dict comprehension](https://docs.python.org/3.13/tutorial/datastructures.html#dictionaries) | `{k: v.copy() for k, v in d.items()}` copies a whole parameter dict | `gradient_check`, `update_parameters` |
| [`np.ndindex(shape)`](https://numpy.org/doc/stable/reference/generated/numpy.ndindex.html) | Iterates over every index tuple of a shape such as `(2, 5)`: `(0, 0), (0, 1), ...` | `gradient_check`, to nudge one parameter at a time |
| [`ndarray.shape`](https://numpy.org/doc/stable/reference/generated/numpy.ndarray.shape.html) | Tuple of sizes: `X.shape[0]` is the sample count $m$, `X.shape[1]` the feature count | `backward`, `train` |
| [`float(x)`](https://docs.python.org/3.13/library/functions.html#float) | Converts a NumPy scalar to a Python `float` | `compute_loss`, `gradient_check`, `accuracy` |
| [`range(n)`](https://docs.python.org/3.13/library/stdtypes.html#range) | Produces `0, 1, ..., n-1` | `train` |
| [`list.append(x)`](https://docs.python.org/3.13/tutorial/datastructures.html#more-on-lists) | Adds an element to the end of a list | `train`, to record the loss |
| `epoch % log_every == 0` | A remainder of 0 means this is a logging step | `train` |

### 2.6 Plotting (provided, just read it)

| API (official docs) | What it does |
| --- | --- |
| [`np.meshgrid`](https://numpy.org/doc/stable/reference/generated/numpy.meshgrid.html), [`np.linspace`](https://numpy.org/doc/stable/reference/generated/numpy.linspace.html) | Builds a grid of points over the plane |
| [`np.c_`](https://numpy.org/doc/stable/reference/generated/numpy.c_.html), [`ndarray.ravel`](https://numpy.org/doc/stable/reference/generated/numpy.ravel.html) | Flattens the grid into a `(k, 2)` sample matrix |
| [`plt.subplots`](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.subplots.html) | Creates a figure and its subplots |
| [`Axes.contourf`](https://matplotlib.org/stable/api/_as_gen/matplotlib.axes.Axes.contourf.html), [`Axes.contour`](https://matplotlib.org/stable/api/_as_gen/matplotlib.axes.Axes.contour.html) | Draws the filled probability map and the $\hat y = 0.5$ contour (the decision boundary) |
| [`Axes.scatter`](https://matplotlib.org/stable/api/_as_gen/matplotlib.axes.Axes.scatter.html), [`Axes.plot`](https://matplotlib.org/stable/api/_as_gen/matplotlib.axes.Axes.plot.html) | Scatter plots, line plots |
| [`Axes.set_yscale("log")`](https://matplotlib.org/stable/api/_as_gen/matplotlib.axes.Axes.set_yscale.html) | Log-scale y-axis, so late-stage loss changes stay visible |

### 2.7 Self-checks

| API (official docs) | What it does |
| --- | --- |
| [`np.testing.assert_allclose(actual, desired, atol=...)`](https://numpy.org/doc/stable/reference/generated/numpy.testing.assert_allclose.html) | Compares arrays within a tolerance and shows the difference on failure |
| [`assert`](https://docs.python.org/3.13/reference/simple_stmts.html#the-assert-statement) | Raises `AssertionError` when the condition is false |
| [`raise ValueError(msg)`](https://docs.python.org/3.13/library/exceptions.html#ValueError) | Stops the function with an error message; `accuracy` uses it when the two shapes differ |

In [ ]:
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split

print("NumPy:", np.__version__)
print("Matplotlib:", matplotlib.__version__)

# Shared plot style: blue / orange tell the two classes apart, and different marker shapes act as a second cue.
CLASS_COLORS = {0: "#2a78d6", 1: "#eb6834"}
CLASS_MARKERS = {0: "o", 1: "^"}
SERIES_COLORS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#4a3aa7"]
# Probability background: 0 → blue, 0.5 → neutral gray, 1 → orange.
PROBA_CMAP = LinearSegmentedColormap.from_list(
    "proba", ["#2a78d6", "#9cc3f0", "#f0efec", "#f6b89c", "#eb6834"]
)
# Hidden-unit activation strength: one hue, light to dark.
ACTIVATION_CMAP = LinearSegmentedColormap.from_list("activation", ["#fcfcfb", "#2a78d6", "#0d3a73"])

plt.rcParams.update({
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titlesize": 11,
})

## 3. Core Hyperparameters

These are the settings for the first run. They are a starting point, and they do not give the best result. Section 7 asks you to change them and watch what happens.

| Hyperparameter | Code constant | Initial value | Notes |
| --- | --- | --- | --- |
| Input features | `N_INPUT` | 2 | $x_1, x_2$ |
| Hidden units | `N_HIDDEN` | 5 | Width of the hidden layer |
| Output units | `N_OUTPUT` | 1 | One probability |
| Hidden activation | (none) | ReLU | The extension task in Section 7 swaps in Tanh |
| Output activation | (none) | Sigmoid | Binary output |
| Loss | (none) | Binary Cross-Entropy | Classification task |
| Learning rate $\alpha$ | `LEARNING_RATE` | 0.05 | How far each step moves along the negative gradient |
| Epochs | `EPOCHS` | 5,000 | In Batch GD each epoch performs exactly one parameter update |
| Loss logging interval | `LOG_EVERY` | 100 | Record the loss every 100 epochs |
| Optimizer | (none) | Batch Gradient Descent | Every update uses all 800 training samples |
| Weight initialization | (none) | He Initialization | Suited to ReLU |
| Bias initialization | (none) | Zeros | Standard practice |
| Regularization | (none) | None | Not used in the first version |
| Random seed | `SEED` | 42 | Makes experiments reproducible |
| Gradient check $\varepsilon$ | `GRAD_CHECK_EPSILON` | $10^{-5}$ | Step size for the centered difference |

The model has $2\times5 + 5 + 5\times1 + 1 = 21$ trainable parameters.

In [ ]:
# Data settings
N_SAMPLES = 1000
NOISE = 0.2
TEST_SIZE = 0.2
SEED = 42

# Network architecture
N_INPUT = 2
N_HIDDEN = 5
N_OUTPUT = 1

# Optimization settings
LEARNING_RATE = 0.05
EPOCHS = 5000
LOG_EVERY = 100

# Gradient checking
GRAD_CHECK_EPSILON = 1e-5

n_params = N_INPUT * N_HIDDEN + N_HIDDEN + N_HIDDEN * N_OUTPUT + N_OUTPUT
print("Trainable parameters:", n_params)
assert n_params == 21

## 4. Formula Sheet

Come back to this section whenever you need it while implementing. Each Lab also repeats the formulas it uses at the top.

### 4.1 Notation and shapes

$m$ is the number of samples ($m=800$ during training). **Each row is one sample**, so the input is written $XW$, not $WX$.

| Math symbol | Code variable | Shape | Meaning |
| --- | --- | --- | --- |
| $X$ | `X` | $(m, 2)$ | Input (standardized) |
| $Y$ | `Y` | $(m, 1)$ | True labels, 0 or 1 |
| $W^{[1]}$ | `parameters["W1"]` | $(2, 5)$ | Hidden weights |
| $b^{[1]}$ | `parameters["b1"]` | $(1, 5)$ | Hidden bias |
| $Z^{[1]}$ | `cache["Z1"]` | $(m, 5)$ | Hidden-layer linear output |
| $A^{[1]}$ | `cache["A1"]` | $(m, 5)$ | Hidden-layer activations |
| $W^{[2]}$ | `parameters["W2"]` | $(5, 1)$ | Output weights |
| $b^{[2]}$ | `parameters["b2"]` | $(1, 1)$ | Output bias |
| $Z^{[2]}$ | `cache["Z2"]` | $(m, 1)$ | Output-layer linear output (logit) |
| $\hat Y$ | `cache["Y_hat"]` | $(m, 1)$ | Probability of Class 1 |
| $dW^{[l]}, db^{[l]}$ | `grads["dW1"]`, etc. | Same as the matching parameter | Gradient of the average loss with respect to that parameter |

### 4.2 Feature standardization (training-set statistics only)

$$
\mu_j=\frac{1}{m_{\text{train}}}\sum_{i=1}^{m_{\text{train}}}x_{ij},\qquad
\sigma_j=\sqrt{\frac{1}{m_{\text{train}}}\sum_{i=1}^{m_{\text{train}}}\left(x_{ij}-\mu_j\right)^2},\qquad
\tilde x_{ij}=\frac{x_{ij}-\mu_j}{\sigma_j}
$$

The test set is transformed with the **training set's** $\mu$ and $\sigma$.

### 4.3 He Initialization

$$
W^{[1]}\sim\mathcal N\!\left(0,\frac{2}{n_{\text{in}}}\right)\ \text{with}\ n_{\text{in}}=2,\qquad
W^{[2]}\sim\mathcal N\!\left(0,\frac{2}{n_{\text{in}}}\right)\ \text{with}\ n_{\text{in}}=5,\qquad
b^{[1]}=0,\ b^{[2]}=0
$$

In code: `np.random.randn(n_in, n_out) * np.sqrt(2 / n_in)`. The second argument of $\mathcal N(0, \sigma^2)$ is the **variance**, so you multiply by the standard deviation $\sqrt{2/n_{\text{in}}}$.

### 4.4 Activation functions and their derivatives

$$
\operatorname{ReLU}(z)=\max(0,z),\qquad
\operatorname{ReLU}'(z)=\begin{cases}1,& z>0\\0,& z\le 0\end{cases}
$$

$$
\sigma(z)=\frac{1}{1+e^{-z}},\qquad \sigma'(z)=\sigma(z)\bigl(1-\sigma(z)\bigr)
$$

ReLU has no derivative at $z=0$. This exercise uses 0 there.

### 4.5 Forward Propagation

$$
Z^{[1]}=XW^{[1]}+b^{[1]},\qquad A^{[1]}=\operatorname{ReLU}(Z^{[1]})
$$

$$
Z^{[2]}=A^{[1]}W^{[2]}+b^{[2]},\qquad \hat Y=\sigma(Z^{[2]})
$$

$b^{[1]}$ has shape $(1,5)$ and is added to every row of the $(m,5)$ matrix through broadcasting.

### 4.6 Binary Cross-Entropy

$$
J=-\frac1m\sum_{i=1}^{m}\Bigl[y_i\log(\hat y_i)+(1-y_i)\log(1-\hat y_i)\Bigr]
$$

Clip the probabilities to $[10^{-12},\,1-10^{-12}]$ before taking logs, to avoid $\log 0$.

### 4.7 Backpropagation

$$
\begin{aligned}
dZ^{[2]} &= \hat Y - Y && (m,1)\\
dW^{[2]} &= \tfrac{1}{m}\,{A^{[1]}}^{\top} dZ^{[2]} && (5,m)(m,1)\to(5,1)\\
db^{[2]} &= \tfrac{1}{m}\textstyle\sum_{i=1}^{m} dZ^{[2]}_{i,:} && (1,1)\\
dA^{[1]} &= dZ^{[2]}\,{W^{[2]}}^{\top} && (m,1)(1,5)\to(m,5)\\
dZ^{[1]} &= dA^{[1]} \odot \operatorname{ReLU}'(Z^{[1]}) && (m,5)\\
dW^{[1]} &= \tfrac{1}{m}\,X^{\top} dZ^{[1]} && (2,m)(m,5)\to(2,5)\\
db^{[1]} &= \tfrac{1}{m}\textstyle\sum_{i=1}^{m} dZ^{[1]}_{i,:} && (1,5)
\end{aligned}
$$

- $\odot$ is element-wise multiplication, written `*` in code; matrix multiplication is `@`.
- $\sum_i$ sums over samples (`axis=0`) and keeps the result 2-D.
- Here $dZ^{[2]}$, $dA^{[1]}$ and $dZ^{[1]}$ are **per-sample** derivatives (one row per sample) and are not divided by $m$. The $\tfrac1m$ is applied exactly once, when computing $dW$ and $db$. If you already divided by $m$ at $dZ^{[2]}$, do not divide again later.

<details>
<summary><b>Derivation (click to expand)</b></summary>

Start with a single sample. Let $\ell=-\bigl[y\log\hat y+(1-y)\log(1-\hat y)\bigr]$ and $\hat y=\sigma(z^{[2]})$.

1. Loss with respect to the probability: $\dfrac{\partial \ell}{\partial \hat y}=-\dfrac{y}{\hat y}+\dfrac{1-y}{1-\hat y}=\dfrac{\hat y-y}{\hat y(1-\hat y)}$.
2. Sigmoid with respect to the logit: $\dfrac{\partial \hat y}{\partial z^{[2]}}=\hat y(1-\hat y)$.
3. Multiply them and the denominator cancels: $\dfrac{\partial \ell}{\partial z^{[2]}}=\hat y-y$. This is one row of $dZ^{[2]}$.
4. Since $z^{[2]}=\sum_j a^{[1]}_j W^{[2]}_j+b^{[2]}$, we get $\dfrac{\partial z^{[2]}}{\partial W^{[2]}_j}=a^{[1]}_j$, $\dfrac{\partial z^{[2]}}{\partial b^{[2]}}=1$, $\dfrac{\partial z^{[2]}}{\partial a^{[1]}_j}=W^{[2]}_j$.
5. Since $a^{[1]}_j=\operatorname{ReLU}(z^{[1]}_j)$, we get $\dfrac{\partial a^{[1]}_j}{\partial z^{[1]}_j}=\operatorname{ReLU}'(z^{[1]}_j)$.
6. Since $z^{[1]}_j=\sum_k x_k W^{[1]}_{kj}+b^{[1]}_j$, we get $\dfrac{\partial z^{[1]}_j}{\partial W^{[1]}_{kj}}=x_k$ and $\dfrac{\partial z^{[1]}_j}{\partial b^{[1]}_j}=1$.
7. Multiply these factors along the chain rule and average over the $m$ samples to get the matrix form above. The matrix product $X^{\top}dZ^{[1]}$ already does the "sum over samples" step, so you only need to divide by $m$ afterwards.

**Shape test**: a gradient must have exactly the same shape as its parameter. A transpose in the wrong place usually makes the shapes disagree.

</details>

### 4.8 Gradient Descent update

$$
W^{[l]}\leftarrow W^{[l]}-\alpha\,dW^{[l]},\qquad b^{[l]}\leftarrow b^{[l]}-\alpha\,db^{[l]},\qquad l\in\{1,2\}
$$

### 4.9 Numerical Gradient Checking

For each parameter element $\theta_k$, nudge it by $\varepsilon$ in each direction while keeping every other parameter fixed:

$$
g^{\text{num}}_k=\frac{J(\theta_k+\varepsilon)-J(\theta_k-\varepsilon)}{2\varepsilon}
$$

Then compare the numerical gradient $g^{\text{num}}$ with the analytic gradient $g^{\text{ana}}$ from `backward`:

$$
\text{relative error}=\frac{\lVert g^{\text{num}}-g^{\text{ana}}\rVert_2}{\lVert g^{\text{num}}\rVert_2+\lVert g^{\text{ana}}\rVert_2+10^{-12}}
$$

| Relative error | Interpretation |
| --- | --- |
| $< 10^{-7}$ | The gradient is correct |
| $10^{-7}$ to $10^{-4}$ | Worth checking. It can also happen when some $Z^{[1]}$ is very close to ReLU's kink at 0: the $\pm\varepsilon$ nudge then crosses the kink, and the centered difference is no longer accurate |
| $> 10^{-4}$ | `backward` almost certainly has a bug |

### 4.10 Prediction and accuracy

$$
\hat y^{\text{class}}_i=\begin{cases}1,&\hat y_i\ge 0.5\\0,&\hat y_i<0.5\end{cases},\qquad
\text{Accuracy}=\frac1m\sum_{i=1}^{m}\mathbb 1\!\left[\hat y^{\text{class}}_i=y_i\right]
$$

### 4.11 Decision boundary

The decision boundary is the set of points where $\hat y(x)=0.5$, which is the same as $z^{[2]}(x)=0$:

$$
z^{[2]}(x)=\sum_{j=1}^{5}W^{[2]}_j\,\operatorname{ReLU}\!\bigl(x\,W^{[1]}_{:,j}+b^{[1]}_j\bigr)+b^{[2]}=0
$$

- **Logistic Regression**: $xW+b=0$, which is always a straight line.
- **This network**: each hidden neuron adds a "fold" along its own line $x\,W^{[1]}_{:,j}+b^{[1]}_j=0$, so the decision boundary is a chain of straight segments. More hidden neurons means more places to fold. The hidden-unit activation maps in Lab 6 draw these fold lines.

## Lab 1: Dataset Preparation

The data loading and splitting code below uses the settings from Section 1. Do not change the random seed.

`make_moons` returns `y` with shape `(m,)`. We turn it into an `(m, 1)` column vector so it matches the shape of $\hat Y$. This matters: `(m, 1) - (m,)` does not raise an error. It broadcasts into an `(m, m)` matrix, and the loss and gradients then go wrong without any warning.

**You implement**: `standardize_features(X_train, X_test)`

$$
\mu=\operatorname{mean}(X_{\text{train}}),\quad \sigma=\operatorname{std}(X_{\text{train}}),\quad
\tilde X_{\text{train}}=\frac{X_{\text{train}}-\mu}{\sigma},\quad \tilde X_{\text{test}}=\frac{X_{\text{test}}-\mu}{\sigma}
$$

- $\mu$ and $\sigma$ are computed per column and have shape `(1, 2)`.
- Compute $\mu$ and $\sigma$ from the **training set only**. Including the test set would leak test information into training.
- API: `np.mean(..., axis=0, keepdims=True)`, `np.std(..., axis=0, keepdims=True)`.

In [ ]:
# Provided: data generation and split, with the settings from Section 1.
X, y = make_moons(n_samples=N_SAMPLES, noise=NOISE, random_state=SEED)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=TEST_SIZE,
    random_state=SEED,
    stratify=y,
)

# Turn labels from (m,) into (m, 1) column vectors so they match Y_hat.
y_train = y_train.reshape(-1, 1)
y_test = y_test.reshape(-1, 1)

print("X_train:", X_train.shape, " y_train:", y_train.shape)
print("X_test: ", X_test.shape, " y_test: ", y_test.shape)

In [ ]:
def standardize_features(
    X_train: np.ndarray,
    X_test: np.ndarray,
) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """Standardize train and test features using training-set statistics only.

    Parameters
    ----------
    X_train : ndarray, shape (m_train, n_x)
        Raw training features.
    X_test : ndarray, shape (m_test, n_x)
        Raw test features.

    Returns
    -------
    X_train_std : ndarray, shape (m_train, n_x)
    X_test_std : ndarray, shape (m_test, n_x)
    mean : ndarray, shape (1, n_x)
        Per-column mean of the training set.
    std : ndarray, shape (1, n_x)
        Per-column population std of the training set (ddof=0).

    Notes
    -----
    Do not modify the inputs; never use X_test to compute mean or std.
    """
    # TODO: Compute per-column mean and std from X_train, keeping shape (1, n_x).
    # TODO: Transform both X_train and X_test with that same mean/std and return the four arrays in order.
    raise NotImplementedError("TODO Lab 1: implement standardize_features")

In [ ]:
# Lab 1 self-check: a small example you can verify by hand.
A_train = np.array([[1.0, 10.0], [3.0, 14.0], [5.0, 18.0]])
A_test = np.array([[7.0, 22.0]])
A_train_before, A_test_before = A_train.copy(), A_test.copy()

A_train_std, A_test_std, mean_a, std_a = standardize_features(A_train, A_test)
assert mean_a.shape == (1, 2) and std_a.shape == (1, 2), "mean / std should be (1, 2)"
np.testing.assert_allclose(mean_a, [[3.0, 14.0]])
np.testing.assert_allclose(std_a, [[np.sqrt(8 / 3), np.sqrt(32 / 3)]])
np.testing.assert_allclose(A_train_std.mean(axis=0), [0.0, 0.0], atol=1e-12)
np.testing.assert_allclose(A_train_std.std(axis=0), [1.0, 1.0], atol=1e-12)
# The test sample uses training statistics, so it is not pulled back to mean 0.
np.testing.assert_allclose(A_test_std, (A_test - mean_a) / std_a)
assert np.all(A_test_std > 0)
np.testing.assert_array_equal(A_train, A_train_before)
np.testing.assert_array_equal(A_test, A_test_before)
print("Lab 1 standardize_features checks passed.")

In [ ]:
# Standardize the real data and check the expected shapes.
X_train_std, X_test_std, feature_mean, feature_std = standardize_features(X_train, X_test)

assert X_train_std.shape == (800, 2)
assert X_test_std.shape == (200, 2)
assert y_train.shape == (800, 1) and y_test.shape == (200, 1)
assert set(np.unique(y)) == {0, 1}
np.testing.assert_allclose(X_train_std.mean(axis=0), [0.0, 0.0], atol=1e-10)
np.testing.assert_allclose(X_train_std.std(axis=0), [1.0, 1.0], atol=1e-10)

labels, counts = np.unique(y_train, return_counts=True)
print("Training mean:", feature_mean.ravel())
print("Training std:", feature_std.ravel())
print("Training class counts:", dict(zip(labels.tolist(), counts.tolist())))
print("Data checks passed.")

In [ ]:
# Plot 1: raw data (train / test) and the standardized training data.
def scatter_classes(ax, X_points, Y_points, size=18, alpha=0.8, label_suffix=""):
    """Scatter points by class; color and marker shape both encode the class."""
    Y_flat = Y_points.ravel()
    for cls in (0, 1):
        mask = Y_flat == cls
        ax.scatter(
            X_points[mask, 0], X_points[mask, 1],
            s=size, alpha=alpha, c=CLASS_COLORS[cls], marker=CLASS_MARKERS[cls],
            edgecolors="white", linewidths=0.4, label=f"Class {cls}{label_suffix}",
        )


fig, axes = plt.subplots(1, 3, figsize=(15, 4.4))
scatter_classes(axes[0], X_train, y_train)
axes[0].set_title("Raw training set (800)")
scatter_classes(axes[1], X_test, y_test)
axes[1].set_title("Raw test set (200)")
scatter_classes(axes[2], X_train_std, y_train)
axes[2].set_title("Standardized training set")
for ax, unit in zip(axes, ["raw", "raw", "standardized"]):
    ax.set_xlabel(f"x1 ({unit})")
    ax.set_ylabel(f"x2 ({unit})")
    ax.legend(loc="upper right", fontsize=8)
fig.tight_layout()
plt.show()

## Lab 2: Parameter Initialization

**You implement**: `initialize_parameters(n_input, n_hidden, n_output, seed=42)`

$$
W^{[1]}=\texttt{randn}(n_{\text{input}},n_{\text{hidden}})\cdot\sqrt{\tfrac{2}{n_{\text{input}}}},\qquad
W^{[2]}=\texttt{randn}(n_{\text{hidden}},n_{\text{output}})\cdot\sqrt{\tfrac{2}{n_{\text{hidden}}}},\qquad
b^{[1]}=\mathbf 0_{(1,n_{\text{hidden}})},\ b^{[2]}=\mathbf 0_{(1,n_{\text{output}})}
$$

Conventions:

1. When `seed` is not `None`, call `np.random.seed(seed)` first, then generate **`W1` first and `W2` second**. A fixed order is what makes the result reproducible.
2. Return a dict with the keys `"W1"`, `"b1"`, `"W2"`, `"b2"`.
3. Each layer's $n_{\text{in}}$ is that layer's input width: `n_input` for the first layer, `n_hidden` for the second.

**Why not set all the weights to 0?** If every hidden neuron starts with the same weights, they compute the same outputs and the same gradients, and they stay identical after every update. Five neurons collapse into one. Random initialization breaks that symmetry. Biases can start at 0 because the weights already differ.

**Why multiply by $\sqrt{2/n_{\text{in}}}$?** Each unit adds up $n_{\text{in}}$ weighted inputs, so the variance of $Z$ grows with $n_{\text{in}}$. Dividing the weight variance by $n_{\text{in}}$ cancels that growth. ReLU then sets about half of the values to 0, and the factor 2 makes up for that lost half. Together, the signal's variance stays about the same from layer to layer. It does not shrink toward 0 or grow very large.

In [ ]:
def initialize_parameters(
    n_input: int,
    n_hidden: int,
    n_output: int,
    seed: int | None = 42,
) -> dict[str, np.ndarray]:
    """Initialize weights with He Initialization and biases with zeros.

    Parameters
    ----------
    n_input : int
        Number of input features (2 here).
    n_hidden : int
        Number of hidden units (5 here).
    n_output : int
        Number of output units (1 here).
    seed : int or None
        If not None, call np.random.seed(seed) first.

    Returns
    -------
    parameters : dict
        "W1": (n_input, n_hidden)   std sqrt(2 / n_input)
        "b1": (1, n_hidden)         zeros
        "W2": (n_hidden, n_output)  std sqrt(2 / n_hidden)
        "b2": (1, n_output)         zeros

    Notes
    -----
    Generate W1 before W2.
    """
    # TODO: Set the random seed (when seed is not None).
    # TODO: Generate W1 then W2 with np.random.randn times the matching He scale.
    # TODO: Create 2-D b1 and b2 with np.zeros and return the parameter dict.
    raise NotImplementedError("TODO Lab 2: implement initialize_parameters")

In [ ]:
# Lab 2 self-check
params_check = initialize_parameters(N_INPUT, N_HIDDEN, N_OUTPUT, seed=SEED)
assert set(params_check) == {"W1", "b1", "W2", "b2"}, "keys should be W1, b1, W2, b2"
expected_shapes = {"W1": (2, 5), "b1": (1, 5), "W2": (5, 1), "b2": (1, 1)}
for name, shape in expected_shapes.items():
    assert params_check[name].shape == shape, f"{name}: {params_check[name].shape} != {shape}"
np.testing.assert_array_equal(params_check["b1"], 0.0)
np.testing.assert_array_equal(params_check["b2"], 0.0)
assert sum(p.size for p in params_check.values()) == 21

# Same seed gives the same result; a different seed gives a different one.
again = initialize_parameters(N_INPUT, N_HIDDEN, N_OUTPUT, seed=SEED)
np.testing.assert_array_equal(params_check["W1"], again["W1"])
other = initialize_parameters(N_INPUT, N_HIDDEN, N_OUTPUT, seed=SEED + 1)
assert not np.allclose(params_check["W1"], other["W1"])

# Generation order: W1 must be the first random numbers drawn after seeding.
np.random.seed(SEED)
np.testing.assert_allclose(params_check["W1"], np.random.randn(2, 5) * np.sqrt(2 / 2))

# Check the scaling with a wide layer: the sample std should be close to sqrt(2 / n_in).
wide = initialize_parameters(400, 300, 200, seed=0)
np.testing.assert_allclose(wide["W1"].std(), np.sqrt(2 / 400), rtol=0.05)
np.testing.assert_allclose(wide["W2"].std(), np.sqrt(2 / 300), rtol=0.05)
print("Lab 2 initialize_parameters checks passed.")
print("W1 =\n", params_check["W1"])
print("W2 =\n", params_check["W2"])

In [ ]:
# Plot 2: does the He-initialized weight distribution match the theoretical normal N(0, 2/n_in)?
n_in_demo = 500
W_demo = initialize_parameters(n_in_demo, 500, 1, seed=0)["W1"].ravel()
sigma_demo = np.sqrt(2 / n_in_demo)

fig, ax = plt.subplots(figsize=(7, 3.8))
ax.hist(W_demo, bins=80, density=True, color=SERIES_COLORS[0], alpha=0.55, label="Initialized weights")
grid_w = np.linspace(-4 * sigma_demo, 4 * sigma_demo, 300)
ax.plot(grid_w, np.exp(-grid_w**2 / (2 * sigma_demo**2)) / (sigma_demo * np.sqrt(2 * np.pi)),
        color="#0b0b0b", linewidth=2, label=f"N(0, 2/{n_in_demo})")
ax.set_title(f"He initialization: sample std = {W_demo.std():.4f}, theory = {sigma_demo:.4f}")
ax.set_xlabel("Weight value")
ax.set_ylabel("Density")
ax.legend()
fig.tight_layout()
plt.show()

## Lab 3: Forward Propagation

### 3.1 Activation functions

**You implement**: `relu(Z)`, `relu_derivative(Z)`, `sigmoid(Z)`

$$
\operatorname{ReLU}(z)=\max(0,z),\qquad
\operatorname{ReLU}'(z)=\begin{cases}1,& z>0\\0,& z\le 0\end{cases},\qquad
\sigma(z)=\frac{1}{1+e^{-z}}
$$

- All three work **element-wise**: the output has the same shape as the input, with no Python loops.
- `relu_derivative` returns a float array (0.0 or 1.0) and uses 0 at $z=0$. It is used by `backward` in Lab 4; it lives here because it pairs with `relu`.
- API: `np.maximum`, a comparison + `.astype(float)`, `np.exp`.
- **Optional**: when $z$ is a large negative number (e.g. $-1000$), `np.exp(-z)` overflows and NumPy warns. You can split by sign: use $\frac{1}{1+e^{-z}}$ for $z\ge0$ and the equivalent $\frac{e^{z}}{1+e^{z}}$ for $z<0$. Compute each formula only on its own elements (boolean mask indexing). `np.where(cond, a, b)` computes both `a` and `b` on every element, so it still overflows. The basic self-check does not require this.

In [ ]:
def relu(Z: np.ndarray) -> np.ndarray:
    """Element-wise max(0, Z). Returns a new array with the same shape as Z; does not modify Z."""
    # TODO: One line is enough.
    raise NotImplementedError("TODO Lab 3: implement relu")


def relu_derivative(Z: np.ndarray) -> np.ndarray:
    """Derivative of ReLU with respect to Z: 1.0 where Z > 0, else 0.0 (including Z == 0).

    Returns a float array with the same shape as Z.
    """
    # TODO: Build a boolean array with a comparison, then convert it to float.
    raise NotImplementedError("TODO Lab 3: implement relu_derivative")


def sigmoid(Z: np.ndarray) -> np.ndarray:
    """Element-wise 1 / (1 + exp(-Z)). Returns an array with the same shape as Z, values in [0, 1]."""
    # TODO: Implement the full sigmoid formula.
    # TODO (optional): Stay numerically stable for large negative inputs.
    raise NotImplementedError("TODO Lab 3: implement sigmoid")

In [ ]:
# Lab 3.1 self-check
Z_act = np.array([[-2.0, 0.0, 3.0], [1.5, -0.5, 0.0]])
Z_act_before = Z_act.copy()

np.testing.assert_array_equal(relu(Z_act), [[0.0, 0.0, 3.0], [1.5, 0.0, 0.0]])
d_act = relu_derivative(Z_act)
assert d_act.shape == Z_act.shape
assert np.issubdtype(d_act.dtype, np.floating), "relu_derivative should return a float array"
np.testing.assert_array_equal(d_act, [[0.0, 0.0, 1.0], [1.0, 0.0, 0.0]])

S_act = sigmoid(np.array([[-2.0], [0.0], [2.0]]))
assert S_act.shape == (3, 1)
np.testing.assert_allclose(S_act[1, 0], 0.5, atol=1e-12)
np.testing.assert_allclose(S_act[0, 0] + S_act[2, 0], 1.0, atol=1e-12)
np.testing.assert_allclose(S_act[2, 0], 0.8807970779778823, atol=1e-12)
assert sigmoid(np.zeros((2, 3))).shape == (2, 3)
np.testing.assert_array_equal(Z_act, Z_act_before)
print("Lab 3.1 activation checks passed.")

# Optional check: prints a message only, never fails the notebook.
with np.errstate(over="raise"):
    try:
        extreme = sigmoid(np.array([[-1000.0], [1000.0]]))
        print("Optional: sigmoid is numerically stable on extreme inputs ✓", extreme.ravel())
    except FloatingPointError:
        print("Optional: sigmoid(-1000) overflows. You can switch to a stable version (later Labs work either way).")

In [ ]:
# Plot 3: the activation functions and their derivatives.
z_line = np.linspace(-5, 5, 401).reshape(-1, 1)
s_line = sigmoid(z_line)

fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
axes[0].plot(z_line, relu(z_line), color=SERIES_COLORS[0], linewidth=2, label="ReLU(z)")
axes[0].plot(z_line, relu_derivative(z_line), color=SERIES_COLORS[1], linewidth=2, linestyle="--", label="ReLU'(z)")
axes[0].set_title("ReLU: gradient is 1 for z > 0 and 0 for z < 0")
axes[1].plot(z_line, s_line, color=SERIES_COLORS[0], linewidth=2, label="sigmoid(z)")
axes[1].plot(z_line, s_line * (1 - s_line), color=SERIES_COLORS[1], linewidth=2, linestyle="--", label="sigmoid'(z) = s(1 - s)")
axes[1].axhline(0.5, color="#52514e", linewidth=1, linestyle=":")
axes[1].set_title("Sigmoid: the derivative peaks at only 0.25 (at z = 0)")
for ax in axes:
    ax.set_xlabel("z")
    ax.legend()
fig.tight_layout()
plt.show()

### 3.2 Forward Propagation

**You implement**: `forward(X, parameters)`

$$
Z^{[1]}=XW^{[1]}+b^{[1]},\qquad A^{[1]}=\operatorname{ReLU}(Z^{[1]}),\qquad
Z^{[2]}=A^{[1]}W^{[2]}+b^{[2]},\qquad \hat Y=\sigma(Z^{[2]})
$$

| Step | Shape reasoning |
| --- | --- |
| $XW^{[1]}$ | $(m,2)\,@\,(2,5)\to(m,5)$ |
| $+\,b^{[1]}$ | $(m,5)+(1,5)\to(m,5)$ (broadcasting) |
| $A^{[1]}W^{[2]}$ | $(m,5)\,@\,(5,1)\to(m,1)$ |
| $+\,b^{[2]}$ | $(m,1)+(1,1)\to(m,1)$ |

- Return `(Y_hat, cache)`, where `cache = {"Z1": Z1, "A1": A1, "Z2": Z2, "Y_hat": Y_hat}`. `backward` needs these intermediate results.
- Call your own `relu` and `sigmoid`.
- Do not compute the loss or apply the 0.5 threshold here.

In [ ]:
def forward(
    X: np.ndarray,
    parameters: dict[str, np.ndarray],
) -> tuple[np.ndarray, dict[str, np.ndarray]]:
    """Forward propagation through the two-layer network.

    Parameters
    ----------
    X : ndarray, shape (m, n_x)
        A batch of samples, one per row.
    parameters : dict
        The dict from initialize_parameters: W1, b1, W2, b2.

    Returns
    -------
    Y_hat : ndarray, shape (m, 1)
        Probability of Class 1.
    cache : dict
        "Z1" (m, n_h), "A1" (m, n_h), "Z2" (m, 1), "Y_hat" (m, 1)

    Notes
    -----
    Does not modify X or parameters and does not read globals.
    """
    # TODO: Unpack the four parameters.
    # TODO: Hidden layer: linear step + ReLU.
    # TODO: Output layer: linear step + sigmoid.
    # TODO: Store the four intermediate results in cache and return (Y_hat, cache).
    raise NotImplementedError("TODO Lab 3: implement forward")

In [ ]:
# Lab 3.2 self-check: 4 samples and 3 hidden units, small enough to compute by hand.
# The sample count (4) and hidden width (3) differ on purpose, so a misplaced transpose shows up as a shape error.
X_hand = np.array([[1.0, 2.0], [-1.0, 0.5], [0.0, -1.0], [2.0, -0.5]])
Y_hand = np.array([[1], [0], [1], [0]])
params_hand = {
    "W1": np.array([[1.0, -1.0, 0.5], [0.5, 1.0, -1.0]]),
    "b1": np.array([[0.0, 0.5, -0.5]]),
    "W2": np.array([[1.0], [-1.0], [2.0]]),
    "b2": np.array([[0.5]]),
}
params_hand_before = {k: v.copy() for k, v in params_hand.items()}

Y_hat_hand, cache_hand = forward(X_hand, params_hand)
assert set(cache_hand) == {"Z1", "A1", "Z2", "Y_hat"}, "cache keys should be Z1, A1, Z2, Y_hat"
assert cache_hand["Z1"].shape == (4, 3) and cache_hand["A1"].shape == (4, 3)
assert cache_hand["Z2"].shape == (4, 1) and Y_hat_hand.shape == (4, 1)

# Row 1: Z1 = [1*1 + 2*0.5 + 0, 1*(-1) + 2*1 + 0.5, 1*0.5 + 2*(-1) - 0.5] = [2, 1.5, -2]
np.testing.assert_allclose(cache_hand["Z1"], [[2.0, 1.5, -2.0], [-0.75, 2.0, -1.5], [-0.5, -0.5, 0.5], [1.75, -2.0, 1.0]])
np.testing.assert_allclose(cache_hand["A1"], [[2.0, 1.5, 0.0], [0.0, 2.0, 0.0], [0.0, 0.0, 0.5], [1.75, 0.0, 1.0]])
# Row 1: Z2 = 2*1 + 1.5*(-1) + 0*2 + 0.5 = 1.0
np.testing.assert_allclose(cache_hand["Z2"], [[1.0], [-1.5], [1.5], [4.25]])
np.testing.assert_allclose(Y_hat_hand, 1 / (1 + np.exp(-np.array([[1.0], [-1.5], [1.5], [4.25]]))))
np.testing.assert_allclose(cache_hand["Y_hat"], Y_hat_hand)
for k in params_hand:
    np.testing.assert_array_equal(params_hand[k], params_hand_before[k])
print("Lab 3.2 forward checks passed.")

In [ ]:
# One forward pass over the 800 training samples with the initial parameters.
parameters_init = initialize_parameters(N_INPUT, N_HIDDEN, N_OUTPUT, seed=SEED)
Y_hat_init, cache_init = forward(X_train_std, parameters_init)
for name, value in cache_init.items():
    print(f"{name:6s} shape = {value.shape}")
print("Initial predicted probability: min = "
      f"{Y_hat_init.min():.3f}, mean = {Y_hat_init.mean():.3f}, max = {Y_hat_init.max():.3f}")
print("Fraction of samples that activate each hidden unit (Z1 > 0):",
      np.round((cache_init["Z1"] > 0).mean(axis=0), 3))

## Lab 4: Loss and Backpropagation

### 4.1 Binary Cross-Entropy

**You implement**: `compute_loss(Y, Y_hat, eps=1e-12)`

$$
J=-\frac1m\sum_{i=1}^{m}\Bigl[y_i\log(\hat y_i)+(1-y_i)\log(1-\hat y_i)\Bigr]
$$

- First build a new array with `np.clip(Y_hat, eps, 1 - eps)`; do not modify the `Y_hat` you were given.
- `log` is the natural log, `np.log`.
- Return a Python `float` (convert with `float(...)`), not a `(1, 1)` array.
- Reference value: if every prediction is 0.5, $J=\log 2\approx0.6931$ regardless of the labels.

In [ ]:
def compute_loss(Y: np.ndarray, Y_hat: np.ndarray, eps: float = 1e-12) -> float:
    """Compute the average binary cross-entropy.

    Parameters
    ----------
    Y : ndarray, shape (m, 1)
        True labels, 0 or 1.
    Y_hat : ndarray, shape (m, 1)
        Predicted probabilities, same shape as Y.
    eps : float
        Clipping constant to avoid log(0).

    Returns
    -------
    loss : float
        Average loss over all samples (Python float).
    """
    # TODO: Clip Y_hat into [eps, 1 - eps] as a new array.
    # TODO: Apply the formula, average over samples, and return a Python float.
    raise NotImplementedError("TODO Lab 4: implement compute_loss")

In [ ]:
# Lab 4.1 self-check
y_loss = np.array([[0], [1]])
neutral = compute_loss(y_loss, np.array([[0.5], [0.5]]))
assert isinstance(neutral, float), "compute_loss should return a Python float"
np.testing.assert_allclose(neutral, np.log(2), atol=1e-12)

good = compute_loss(y_loss, np.array([[0.1], [0.9]]))
bad = compute_loss(y_loss, np.array([[0.9], [0.1]]))
assert 0.0 < good < neutral < bad
np.testing.assert_allclose(good, -np.log(0.9), atol=1e-12)

edge = np.array([[1.0], [0.0]])
edge_before = edge.copy()
assert np.isfinite(compute_loss(y_loss, edge)), "the loss must stay finite when predictions are exactly 0 or 1"
np.testing.assert_array_equal(edge, edge_before)

# Reuse the hand-computed example from Lab 3.2.
np.testing.assert_allclose(compute_loss(Y_hand, Y_hat_hand), 1.245062925103806, atol=1e-10)
print("Lab 4.1 compute_loss checks passed.")
print("Training loss with the initial parameters:", compute_loss(y_train, Y_hat_init))

### 4.2 Backpropagation

**You implement**: `backward(X, Y, parameters, cache)`

Work from the output back to the input (full derivation in Section 4.7):

| Order | Formula | Code hint | Shape |
| --- | --- | --- | --- |
| 1 | $dZ^{[2]}=\hat Y-Y$ | `cache["Y_hat"] - Y` | $(m,1)$ |
| 2 | $dW^{[2]}=\frac1m {A^{[1]}}^{\top}dZ^{[2]}$ | `.T`, `@`, `/ m` | $(5,1)$ |
| 3 | $db^{[2]}=\frac1m\sum_i dZ^{[2]}_{i,:}$ | `np.sum(..., axis=0, keepdims=True) / m` | $(1,1)$ |
| 4 | $dA^{[1]}=dZ^{[2]}{W^{[2]}}^{\top}$ | `@`, `.T` | $(m,5)$ |
| 5 | $dZ^{[1]}=dA^{[1]}\odot\operatorname{ReLU}'(Z^{[1]})$ | `*`, `relu_derivative(cache["Z1"])` | $(m,5)$ |
| 6 | $dW^{[1]}=\frac1m X^{\top}dZ^{[1]}$ | `.T`, `@`, `/ m` | $(2,5)$ |
| 7 | $db^{[1]}=\frac1m\sum_i dZ^{[1]}_{i,:}$ | `np.sum(..., axis=0, keepdims=True) / m` | $(1,5)$ |

- Read $m$ from `X.shape[0]`; do not hard-code 800.
- Return the dict `{"dW1", "db1", "dW2", "db2"}`. Each gradient must have exactly the same shape as its parameter.
- Step 5 is element-wise `*`, not matrix `@`. It means that when a neuron was inactive for a sample in the forward pass ($Z^{[1]}\le0$), it passes no gradient back for that sample.
- Only compute gradients here; do not update the parameters.

In [ ]:
def backward(
    X: np.ndarray,
    Y: np.ndarray,
    parameters: dict[str, np.ndarray],
    cache: dict[str, np.ndarray],
) -> dict[str, np.ndarray]:
    """Compute the gradients of the average BCE with respect to the four parameters using the analytic formulas.

    Parameters
    ----------
    X : ndarray, shape (m, n_x)
        The same input that was passed to forward.
    Y : ndarray, shape (m, 1)
        True labels.
    parameters : dict
        W1, b1, W2, b2
    cache : dict
        Z1, A1, Z2, Y_hat from forward (Y_hat is unclipped).

    Returns
    -------
    grads : dict
        "dW1" (n_x, n_h), "db1" (1, n_h), "dW2" (n_h, 1), "db2" (1, 1)

    Notes
    -----
    Does not modify any input and does not update parameters.
    """
    # TODO: Read the sample count m.
    # TODO: Output layer: dZ2, dW2, db2.
    # TODO: Send the error back to the hidden layer: dA1, dZ1 (use relu_derivative).
    # TODO: Hidden layer: dW1, db1, then return the grads dict.
    raise NotImplementedError("TODO Lab 4: implement backward")

In [ ]:
# Lab 4.2 self-check: reuses the hand example from Lab 3.2 (expected values come from the formulas in Section 4.7).
grads_hand = backward(X_hand, Y_hand, params_hand, cache_hand)
assert set(grads_hand) == {"dW1", "db1", "dW2", "db2"}
for name in ("W1", "b1", "W2", "b2"):
    assert grads_hand["d" + name].shape == params_hand[name].shape, (
        f"d{name} has shape {grads_hand['d' + name].shape}, but the parameter has shape {params_hand[name].shape}"
    )

np.testing.assert_allclose(grads_hand["db2"], [[0.17924873789668982]], atol=1e-10)
np.testing.assert_allclose(grads_hand["dW2"], [[0.2968764524835825], [-0.00964027111056999], [0.22368090276339406]], atol=1e-10)
np.testing.assert_allclose(grads_hand["db1"], [[0.17924873789668982, 0.02162897439090969, 0.401755424575199]], atol=1e-10)
np.testing.assert_allclose(
    grads_hand["dW1"],
    [[0.42573283113587845, 0.11284173629408786, 0.9859363729567544],
     [-0.25771275730459187, 0.11166752020920301, -0.15527133133601043]],
    atol=1e-10,
)
for k in params_hand:
    np.testing.assert_array_equal(params_hand[k], params_hand_before[k])
print("Lab 4.2 backward checks passed.")

### 4.3 Numerical Gradient Checking

**You implement**: `gradient_check(parameters, grads, X, Y, epsilon=1e-5)`

The self-check above covers only one example. Numerical gradient checking can test `backward` on **any** parameters and data. It uses only `forward` and `compute_loss`, so it does not depend on the backprop formulas:

$$
g^{\text{num}}_k=\frac{J(\theta_k+\varepsilon)-J(\theta_k-\varepsilon)}{2\varepsilon},\qquad
\text{relative error}=\frac{\lVert g^{\text{num}}-g^{\text{ana}}\rVert_2}{\lVert g^{\text{num}}\rVert_2+\lVert g^{\text{ana}}\rVert_2+10^{-12}}
$$

Steps:

1. For each parameter name `"W1"`, `"b1"`, `"W2"`, `"b2"`, make an `np.zeros_like` array to hold the numerical gradient.
2. Loop over every element position `idx` of that parameter with `np.ndindex(shape)`.
3. Make two copies of the parameter dict (`{k: v.copy() for ...}`): add `epsilon` at `idx` in one and subtract it in the other.
4. Run `forward` and `compute_loss` on each copy and plug the two losses into the centered-difference formula.
5. Compute that parameter's relative error with `np.linalg.norm` and store it in the result dict.

- Return `{"W1": float, "b1": float, "W2": float, "b2": float}`, one relative error per parameter.
- The `grads` argument holds the analytic gradients from `backward`. Passing them in means you can deliberately hand in wrong gradients and confirm the checker catches them.
- **You must copy**: changing `parameters[name][idx]` directly would change the caller's parameters.
- 21 parameters × 2 = 42 forward passes. That is fast on 50 samples, but too slow to repeat every epoch, so do not call this inside the training loop.
- The self-checks below accept relative errors below $10^{-6}$, which leaves some margin. A correct `backward` usually gives about $10^{-10}$.

In [ ]:
def gradient_check(
    parameters: dict[str, np.ndarray],
    grads: dict[str, np.ndarray],
    X: np.ndarray,
    Y: np.ndarray,
    epsilon: float = 1e-5,
) -> dict[str, float]:
    """Verify analytic gradients with centered differences.

    Parameters
    ----------
    parameters : dict
        W1, b1, W2, b2 (left unchanged)
    grads : dict
        Analytic gradients from backward: dW1, db1, dW2, db2.
    X : ndarray, shape (m, n_x)
    Y : ndarray, shape (m, 1)
    epsilon : float
        Centered-difference step size.

    Returns
    -------
    relative_errors : dict
        {"W1": ..., "b1": ..., "W2": ..., "b2": ...}, each a Python float.
    """
    # TODO: Loop over the four parameter names and prepare a numerical array for each.
    # TODO: For each element (np.ndindex): copy the params, apply ±epsilon, compute both losses, apply the centered difference.
    # TODO: Compute the relative error between numerical and grads["d" + name] and store it as a float.
    raise NotImplementedError("TODO Lab 4: implement gradient_check")

In [ ]:
# Lab 4.3 self-check
errors_hand = gradient_check(params_hand, grads_hand, X_hand, Y_hand, epsilon=GRAD_CHECK_EPSILON)
assert set(errors_hand) == {"W1", "b1", "W2", "b2"}
assert all(isinstance(v, float) for v in errors_hand.values()), "each relative error should be a Python float"
assert max(errors_hand.values()) < 1e-6, f"Correct gradients should give tiny errors: {errors_hand}"

# Deliberately double dW1; the checker must catch it.
grads_wrong = {k: v.copy() for k, v in grads_hand.items()}
grads_wrong["dW1"] = grads_wrong["dW1"] * 2
errors_wrong = gradient_check(params_hand, grads_wrong, X_hand, Y_hand, epsilon=GRAD_CHECK_EPSILON)
assert errors_wrong["W1"] > 1e-2, "the wrong dW1 was not detected"
assert errors_wrong["W2"] < 1e-6, "only W1's error should grow"

for k in params_hand:
    np.testing.assert_array_equal(params_hand[k], params_hand_before[k], err_msg="gradient_check must not modify parameters")
print("Lab 4.3 gradient_check checks passed.")
print("Correct gradients:", {k: f"{v:.2e}" for k, v in errors_hand.items()})
print("Wrong gradients:  ", {k: f"{v:.2e}" for k, v in errors_wrong.items()})

In [ ]:
# Gradient check on real data (first 50 training samples) with He-initialized parameters, then plot the result.
X_gc, Y_gc = X_train_std[:50], y_train[:50]
parameters_gc = initialize_parameters(N_INPUT, N_HIDDEN, N_OUTPUT, seed=SEED)
_, cache_gc = forward(X_gc, parameters_gc)
grads_gc = backward(X_gc, Y_gc, parameters_gc, cache_gc)
errors_gc = gradient_check(parameters_gc, grads_gc, X_gc, Y_gc, epsilon=GRAD_CHECK_EPSILON)

names = list(errors_gc)
values = [max(errors_gc[n], 1e-16) for n in names]
fig, ax = plt.subplots(figsize=(7, 3.8))
ax.bar(names, values, color=SERIES_COLORS[0], width=0.55)
ax.axhline(1e-7, color="#0ca30c", linestyle="--", linewidth=1.5, label="1e-7: correct")
ax.axhline(1e-4, color="#d03b3b", linestyle="--", linewidth=1.5, label="1e-4: likely a bug")
ax.set_yscale("log")
ax.set_ylim(1e-14, 1)
for i, v in enumerate(values):
    ax.text(i, v * 2, f"{v:.1e}", ha="center", fontsize=9, color="#52514e")
ax.set_ylabel("Relative error (log scale)")
ax.set_title("Numerical Gradient Check: backward on real data")
ax.legend(loc="upper right")
fig.tight_layout()
plt.show()

assert max(errors_gc.values()) < 1e-6, f"Gradient check failed on real data: {errors_gc}"
print("Gradient check on real data passed.")

## Lab 5: Training Loop

### 5.1 Parameter update

**You implement**: `update_parameters(parameters, grads, learning_rate)`

$$
W^{[l]}\leftarrow W^{[l]}-\alpha\,dW^{[l]},\qquad b^{[l]}\leftarrow b^{[l]}-\alpha\,db^{[l]}
$$

- Return a **new** parameter dict and leave the input arrays unchanged. That way you can compare parameters before and after an update, which makes debugging easier.
- Use a minus sign. The gradient points in the direction where the loss grows fastest, so stepping the opposite way makes the loss go down.
- `backward` already divided the gradients by $m$; do not divide again here.

In [ ]:
def update_parameters(
    parameters: dict[str, np.ndarray],
    grads: dict[str, np.ndarray],
    learning_rate: float,
) -> dict[str, np.ndarray]:
    """Perform one gradient descent step and return a new parameter dict.

    Parameters
    ----------
    parameters : dict
        W1, b1, W2, b2
    grads : dict
        dW1, db1, dW2, db2 (same shapes as the parameters)
    learning_rate : float
        Learning rate alpha, greater than 0.

    Returns
    -------
    new_parameters : dict
        Same keys and shapes as parameters.

    Notes
    -----
    Does not modify the arrays in parameters in place.
    """
    # TODO: For each parameter name, compute the new value with grads["d" + name], put it in a new dict, and return it.
    raise NotImplementedError("TODO Lab 5: implement update_parameters")

In [ ]:
# Lab 5.1 self-check
p_old = {"W1": np.array([[1.0, -2.0]]), "b1": np.array([[0.5, 0.0]]),
         "W2": np.array([[2.0], [1.0]]), "b2": np.array([[-1.0]])}
g_step = {"dW1": np.array([[0.2, -0.4]]), "db1": np.array([[-0.1, 0.3]]),
          "dW2": np.array([[1.0], [0.0]]), "db2": np.array([[0.5]])}
p_old_snapshot = {k: v.copy() for k, v in p_old.items()}

p_new = update_parameters(p_old, g_step, learning_rate=0.1)
assert set(p_new) == set(p_old)
np.testing.assert_allclose(p_new["W1"], [[0.98, -1.96]])
np.testing.assert_allclose(p_new["b1"], [[0.51, -0.03]])
np.testing.assert_allclose(p_new["W2"], [[1.9], [1.0]])
np.testing.assert_allclose(p_new["b2"], [[-1.05]])
for k in p_old:
    np.testing.assert_array_equal(p_old[k], p_old_snapshot[k], err_msg="the input must not be modified in place")
print("Lab 5.1 update_parameters checks passed.")

### 5.2 Training loop

**You implement**: `train(X, Y, n_hidden=5, learning_rate=0.05, epochs=5000, log_every=100, seed=42, verbose=True)`

Each epoch uses all of the training data (Batch Gradient Descent):

```text
parameters = initialize_parameters(X.shape[1], n_hidden, Y.shape[1], seed=seed)   # once, before the loop
for epoch in range(epochs):
    1. Forward: Y_hat, cache = forward(X, parameters)
    2. If epoch % log_every == 0: compute the loss from this Y_hat, append it to history; print it when verbose=True
    3. Backward: grads = backward(X, Y, parameters, cache)
    4. Update: parameters = update_parameters(parameters, grads, learning_rate)
```

**History convention**:

- Return `history = {"epoch": [...], "loss": [...]}`, with both lists the same length.
- Each entry is the loss **before** that epoch's update, so the first entry is always `epoch = 0`, the loss of the initial parameters.
- With `epochs=5000` and `log_every=100` you record 50 entries: `0, 100, ..., 4900`.
- The returned `parameters` are the values after all updates. Their loss has to be computed once more after training, and it is usually a little lower than `history["loss"][-1]`.

**Initialization convention**: initialize with `initialize_parameters(X.shape[1], n_hidden, Y.shape[1], seed=seed)`. Lab 6 uses the same seed to rebuild the epoch-0 parameters when it plots how training evolves.

In [ ]:
def train(
    X: np.ndarray,
    Y: np.ndarray,
    n_hidden: int = 5,
    learning_rate: float = 0.05,
    epochs: int = 5000,
    log_every: int = 100,
    seed: int | None = 42,
    verbose: bool = True,
) -> tuple[dict[str, np.ndarray], dict[str, list]]:
    """Train the two-layer network with batch gradient descent.

    Parameters
    ----------
    X : ndarray, shape (m, n_x)
        Standardized training features.
    Y : ndarray, shape (m, 1)
        Training labels.
    n_hidden : int
        Hidden layer width.
    learning_rate : float
        Learning rate alpha.
    epochs : int
        Number of epochs; one parameter update per epoch.
    log_every : int
        Record the loss every this many epochs.
    seed : int or None
        Passed to initialize_parameters.
    verbose : bool
        If True, print epoch and loss at each logging step.

    Returns
    -------
    parameters : dict
        W1, b1, W2, b2 after training.
    history : dict
        {"epoch": list[int], "loss": list[float]}, losses recorded before each update.

    Notes
    -----
    Reuse the functions above; do not write a second copy of the forward / backward logic here.
    """
    # TODO: Initialize the parameters (once) and history.
    # TODO: Loop for epochs: forward → (at intervals) record loss → backward → update.
    # TODO: When verbose=True, print, e.g. f"epoch {epoch:5d} | loss {loss:.6f}".
    # TODO: Return (parameters, history).
    raise NotImplementedError("TODO Lab 5: implement train")

In [ ]:
# Lab 5.2 self-check: a tiny 6-point dataset, unrelated to the real data.
X_tiny = np.array([[-1.0, -1.0], [-1.0, 1.0], [1.0, -1.0], [1.0, 1.0], [0.5, 0.2], [-0.3, -0.8]])
Y_tiny = np.array([[0], [1], [1], [1], [1], [0]])

# 1 epoch: check when the loss is recorded and that parameters were updated exactly once.
p_one, h_one = train(X_tiny, Y_tiny, n_hidden=4, learning_rate=0.5, epochs=1, log_every=1, seed=0, verbose=False)
p_start = initialize_parameters(2, 4, 1, seed=0)
Y_hat_start, cache_start = forward(X_tiny, p_start)
assert h_one["epoch"] == [0]
np.testing.assert_allclose(h_one["loss"][0], compute_loss(Y_tiny, Y_hat_start))
p_expected = update_parameters(p_start, backward(X_tiny, Y_tiny, p_start, cache_start), 0.5)
for k in p_expected:
    np.testing.assert_allclose(p_one[k], p_expected[k], err_msg=f"after one epoch, {k} is wrong")

# Several epochs: logging steps, length, decreasing loss, reproducibility.
p_tiny, h_tiny = train(X_tiny, Y_tiny, n_hidden=4, learning_rate=0.5, epochs=250, log_every=50, seed=0, verbose=False)
assert h_tiny["epoch"] == [0, 50, 100, 150, 200], h_tiny["epoch"]
assert len(h_tiny["loss"]) == 5
assert all(isinstance(v, float) for v in h_tiny["loss"])
assert h_tiny["loss"][-1] < h_tiny["loss"][0]
p_tiny_again, h_tiny_again = train(X_tiny, Y_tiny, n_hidden=4, learning_rate=0.5, epochs=250, log_every=50, seed=0, verbose=False)
np.testing.assert_allclose(h_tiny["loss"], h_tiny_again["loss"])
print("Sample output with verbose=True:")
train(X_tiny, Y_tiny, n_hidden=4, learning_rate=0.5, epochs=101, log_every=50, seed=0, verbose=True)
print("Lab 5.2 train checks passed.")

### 5.3 Train on the 800 training samples

The cell below trains with the settings from Section 3. Only the training set goes into `train`; the test set waits for Lab 6.

In [ ]:
parameters, history = train(
    X_train_std,
    y_train,
    n_hidden=N_HIDDEN,
    learning_rate=LEARNING_RATE,
    epochs=EPOCHS,
    log_every=LOG_EVERY,
    seed=SEED,
    verbose=True,
)

assert len(history["epoch"]) == len(range(0, EPOCHS, LOG_EVERY))  # 50 with the defaults
assert all(np.isfinite(history["loss"]))
final_train_loss = compute_loss(y_train, forward(X_train_std, parameters)[0])
print(f"\nTraining loss after training: {final_train_loss:.6f}")

In [ ]:
# Plot 4: loss curve. Left is linear, right is log scale, where slow late-stage progress is easier to see.
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, scale in zip(axes, ["linear", "log"]):
    ax.plot(history["epoch"], history["loss"], color=SERIES_COLORS[0], linewidth=2, marker="o", markersize=3)
    ax.set_yscale(scale)
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Training BCE (before update)")
    ax.set_title(f"Loss curve ({scale} scale)")
axes[0].annotate(
    f"last logged: {history['loss'][-1]:.4f}",
    xy=(history["epoch"][-1], history["loss"][-1]),
    xytext=(0.55, 0.6), textcoords="axes fraction",
    arrowprops={"arrowstyle": "->", "color": "#52514e"}, color="#52514e",
)
fig.suptitle(f"lr = {LEARNING_RATE}, hidden = {N_HIDDEN}, epochs = {EPOCHS}", y=1.02)
fig.tight_layout()
plt.show()

## Lab 6: Evaluation and Visualization

### 6.1 Prediction and accuracy

**You implement**: `predict_proba(X, parameters)`, `predict(X, parameters, threshold=0.5)`, `accuracy(Y, Y_pred)`

$$
\hat Y=\text{the first return value of forward}(X),\qquad
\hat y^{\text{class}}_i=\mathbb 1[\hat y_i\ge 0.5],\qquad
\text{Accuracy}=\frac1m\sum_{i=1}^{m}\mathbb 1\!\left[\hat y^{\text{class}}_i=y_i\right]
$$

- `predict_proba` returns probabilities with shape `(m, 1)`. Reuse `forward` directly.
- `predict` returns a column vector of 0/1 **integers**. A probability **exactly equal** to the threshold counts as 1 (use `>=`).
- `accuracy` returns a Python `float` in `[0, 1]`. Check first that the two arrays have the same shape and `raise ValueError` if not, so `(m, 1)` and `(m,)` cannot broadcast silently.
- API: comparisons, `.astype(int)`, `np.mean`, `float`.

In [ ]:
def predict_proba(X: np.ndarray, parameters: dict[str, np.ndarray]) -> np.ndarray:
    """Return the probability of Class 1, shape (m, 1)."""
    # TODO: Reuse forward and return only the probabilities.
    raise NotImplementedError("TODO Lab 6: implement predict_proba")


def predict(X: np.ndarray, parameters: dict[str, np.ndarray], threshold: float = 0.5) -> np.ndarray:
    """Return 0/1 integer labels, shape (m, 1); 1 where probability >= threshold."""
    # TODO: Compare probabilities with the threshold, then convert to int.
    raise NotImplementedError("TODO Lab 6: implement predict")


def accuracy(Y: np.ndarray, Y_pred: np.ndarray) -> float:
    """Return the fraction of correct predictions (Python float in [0, 1]).

    Raise ValueError if Y and Y_pred have different shapes.
    """
    # TODO: Check the shapes first, then compare and average.
    raise NotImplementedError("TODO Lab 6: implement accuracy")

In [ ]:
# Lab 6.1 self-check
proba_hand = predict_proba(X_hand, params_hand)
assert proba_hand.shape == (4, 1)
np.testing.assert_allclose(proba_hand, Y_hat_hand)

pred_hand = predict(X_hand, params_hand)
assert pred_hand.shape == (4, 1)
assert np.issubdtype(pred_hand.dtype, np.integer), "predict should return an integer array"
np.testing.assert_array_equal(pred_hand, [[1], [0], [1], [1]])
np.testing.assert_array_equal(predict(X_hand, params_hand, threshold=0.8), [[0], [0], [1], [1]])

# A probability of exactly 0.5 (Z2 = 0) counts as 1.
params_half = {k: np.zeros_like(v) for k, v in params_hand.items()}
np.testing.assert_array_equal(predict(X_hand, params_half), [[1], [1], [1], [1]])

acc_hand = accuracy(Y_hand, pred_hand)
assert isinstance(acc_hand, float)
np.testing.assert_allclose(acc_hand, 0.75)
try:
    accuracy(Y_hand.ravel(), pred_hand)
except ValueError:
    pass
else:
    raise AssertionError("mismatched label shapes must raise ValueError")
print("Lab 6.1 checks passed.")

In [ ]:
# Evaluate the trained parameters on the training and test sets.
train_proba = predict_proba(X_train_std, parameters)
test_proba = predict_proba(X_test_std, parameters)
train_acc = accuracy(y_train, predict(X_train_std, parameters))
test_acc = accuracy(y_test, predict(X_test_std, parameters))

print(f"Train BCE: {compute_loss(y_train, train_proba):.4f}   Train Accuracy: {train_acc:.2%}")
print(f"Test  BCE: {compute_loss(y_test, test_proba):.4f}   Test  Accuracy: {test_acc:.2%}")

### 6.2 Observation tools (provided)

Everything below is an **observation tool**, not part of the exercise. These helpers only call functions you implemented.

- `plot_decision_boundary(predict_proba_fn, X, Y, ...)`: lays a dense grid over the plane, computes $\hat y$ at every grid point, colors the plane by probability, draws the $\hat y=0.5$ decision boundary as a black line, and circles misclassified samples. It takes any function that maps a `(k, 2)` input to `(k, 1)` probabilities, so it can draw both the neural network and Logistic Regression.
- `train_logistic_regression_baseline(...)`: a hand-written Logistic Regression model with no hidden layer (the same model you built in C1_W3). It is the linear model we compare against. It reuses your `sigmoid` and trains with the same learning rate and number of epochs as the network.

The axes show the standardized features, because those are the inputs the model receives.

In [ ]:
def plot_decision_boundary(
    predict_proba_fn,
    X_points: np.ndarray,
    Y_points: np.ndarray,
    ax=None,
    title: str = "",
    resolution: int = 300,
    padding: float = 0.5,
    highlight_errors: bool = True,
):
    """Draw the probability background, the decision boundary, and the samples.

    predict_proba_fn : callable mapping a (k, 2) input to (k, 1) Class 1 probabilities.
    """
    if ax is None:
        _, ax = plt.subplots(figsize=(6, 5))
    x_min, x_max = X_points[:, 0].min() - padding, X_points[:, 0].max() + padding
    y_min, y_max = X_points[:, 1].min() - padding, X_points[:, 1].max() + padding
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, resolution), np.linspace(y_min, y_max, resolution))
    grid = np.c_[xx.ravel(), yy.ravel()]
    proba = predict_proba_fn(grid).reshape(xx.shape)

    filled = ax.contourf(xx, yy, proba, levels=np.linspace(0, 1, 21), cmap=PROBA_CMAP, alpha=0.75)
    ax.contour(xx, yy, proba, levels=[0.5], colors="#0b0b0b", linewidths=2)
    scatter_classes(ax, X_points, Y_points, size=16, alpha=0.9)

    if highlight_errors:
        wrong = ((predict_proba_fn(X_points) >= 0.5).astype(int) != Y_points).ravel()
        ax.scatter(X_points[wrong, 0], X_points[wrong, 1], s=70, facecolors="none",
                   edgecolors="#0b0b0b", linewidths=1.2, label=f"misclassified ({wrong.sum()})")
    ax.set_xlim(x_min, x_max)
    ax.set_ylim(y_min, y_max)
    ax.set_xlabel("x1 (standardized)")
    ax.set_ylabel("x2 (standardized)")
    ax.set_title(title)
    ax.grid(False)
    ax.legend(loc="lower left", fontsize=8, framealpha=0.85)
    return filled


def train_logistic_regression_baseline(
    X: np.ndarray, Y: np.ndarray, learning_rate: float = LEARNING_RATE, epochs: int = EPOCHS
) -> tuple[np.ndarray, np.ndarray]:
    """Hand-written Logistic Regression baseline (no hidden layer), reusing your sigmoid."""
    m, n_x = X.shape
    W = np.zeros((n_x, 1))
    b = np.zeros((1, 1))
    for _ in range(epochs):
        dZ = sigmoid(X @ W + b) - Y
        W = W - learning_rate * (X.T @ dZ) / m
        b = b - learning_rate * np.sum(dZ, axis=0, keepdims=True) / m
    return W, b


W_lr, b_lr = train_logistic_regression_baseline(X_train_std, y_train)


def lr_predict_proba(X_points: np.ndarray) -> np.ndarray:
    return sigmoid(X_points @ W_lr + b_lr)


def nn_predict_proba(X_points: np.ndarray) -> np.ndarray:
    return predict_proba(X_points, parameters)


lr_train_acc = accuracy(y_train, (lr_predict_proba(X_train_std) >= 0.5).astype(int))
lr_test_acc = accuracy(y_test, (lr_predict_proba(X_test_std) >= 0.5).astype(int))
print(f"Logistic Regression  Train: {lr_train_acc:.2%}   Test: {lr_test_acc:.2%}")
print(f"Two-Layer NN         Train: {train_acc:.2%}   Test: {test_acc:.2%}")

In [ ]:
# Plot 5: decision boundaries of Logistic Regression (a straight line) vs. the Two-Layer NN (a chain of segments).
fig, axes = plt.subplots(2, 2, figsize=(13, 10.5))
plot_decision_boundary(lr_predict_proba, X_train_std, y_train, ax=axes[0, 0],
                       title=f"Logistic Regression · train · acc {lr_train_acc:.1%}")
plot_decision_boundary(lr_predict_proba, X_test_std, y_test, ax=axes[0, 1],
                       title=f"Logistic Regression · test · acc {lr_test_acc:.1%}")
plot_decision_boundary(nn_predict_proba, X_train_std, y_train, ax=axes[1, 0],
                       title=f"Two-Layer NN ({N_HIDDEN} hidden) · train · acc {train_acc:.1%}")
filled = plot_decision_boundary(nn_predict_proba, X_test_std, y_test, ax=axes[1, 1],
                                title=f"Two-Layer NN ({N_HIDDEN} hidden) · test · acc {test_acc:.1%}")
fig.colorbar(filled, ax=axes, shrink=0.6, label="Predicted probability P(y = 1)")
plt.show()

In [ ]:
# Plot 6: what each hidden neuron "sees".
# Each panel shows one neuron's activation A1_j = ReLU(x · W1[:, j] + b1[j]): zero on one side of the dashed line, growing linearly on the other.
# The sign of W2[j] decides whether the neuron pushes the output toward Class 1 (+) or Class 0 (−). The last panel is the combined output probability.
pad = 0.5
gx, gy = np.meshgrid(
    np.linspace(X_train_std[:, 0].min() - pad, X_train_std[:, 0].max() + pad, 250),
    np.linspace(X_train_std[:, 1].min() - pad, X_train_std[:, 1].max() + pad, 250),
)
grid_points = np.c_[gx.ravel(), gy.ravel()]
_, grid_cache = forward(grid_points, parameters)

n_panels = N_HIDDEN + 1
n_cols = 3
n_rows = int(np.ceil(n_panels / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(4.6 * n_cols, 4 * n_rows))
axes = np.atleast_1d(axes).ravel()
for j in range(N_HIDDEN):
    ax = axes[j]
    act = grid_cache["A1"][:, j].reshape(gx.shape)
    ax.contourf(gx, gy, act, levels=20, cmap=ACTIVATION_CMAP)
    if act.max() > 0 and act.min() == 0:
        ax.contour(gx, gy, grid_cache["Z1"][:, j].reshape(gx.shape), levels=[0], colors="#eb6834", linestyles="--", linewidths=1.5)
    scatter_classes(ax, X_train_std, y_train, size=5, alpha=0.35)
    w2 = parameters["W2"][j, 0]
    active = (forward(X_train_std, parameters)[1]["Z1"][:, j] > 0).mean()
    ax.set_title(f"Neuron {j + 1}: W2 = {w2:+.2f}, active on {active:.0%}")
    ax.set_xlabel("x1 (standardized)")
    ax.set_ylabel("x2 (standardized)")
    ax.grid(False)
ax = axes[N_HIDDEN]
ax.contourf(gx, gy, grid_cache["Y_hat"].reshape(gx.shape), levels=np.linspace(0, 1, 21), cmap=PROBA_CMAP)
ax.contour(gx, gy, grid_cache["Y_hat"].reshape(gx.shape), levels=[0.5], colors="#0b0b0b", linewidths=2)
scatter_classes(ax, X_train_std, y_train, size=5, alpha=0.35)
ax.set_title("Output Y_hat (combined)")
ax.set_xlabel("x1 (standardized)")
ax.set_ylabel("x2 (standardized)")
ax.grid(False)
for ax in axes[n_panels:]:
    ax.axis("off")
fig.tight_layout()
plt.show()

In [ ]:
# Plot 7: how the decision boundary changes during training.
# With the same seed, train(epochs=k) is exactly the first k steps of the full run, so we retrain up to each checkpoint.
snapshot_epochs = [0, 100, 500, 1000, 2500, EPOCHS]
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
for ax, k in zip(axes.ravel(), snapshot_epochs):
    if k == 0:
        params_k = initialize_parameters(N_INPUT, N_HIDDEN, N_OUTPUT, seed=SEED)
    else:
        params_k, _ = train(X_train_std, y_train, n_hidden=N_HIDDEN, learning_rate=LEARNING_RATE,
                            epochs=k, log_every=LOG_EVERY, seed=SEED, verbose=False)
    acc_k = accuracy(y_train, predict(X_train_std, params_k))
    loss_k = compute_loss(y_train, predict_proba(X_train_std, params_k))
    plot_decision_boundary(lambda P, p=params_k: predict_proba(P, p), X_train_std, y_train, ax=ax,
                           title=f"epoch {k} · loss {loss_k:.3f} · acc {acc_k:.1%}", highlight_errors=False)
fig.tight_layout()
plt.show()

## 7. Experiments: change the settings and watch what happens

**First, compare your results.** With a correct implementation, the default settings (5 hidden units, lr = 0.05, 5,000 epochs) give a last logged training loss of about 0.255, 87.5% training accuracy, and 91.0% test accuracy. Logistic Regression gets 85.75% and 92.0%, so the network is not much better yet. Its decision boundary has only a few straight pieces and does not follow the moon shapes.

**This is not a bug.** Look at the loss curve: after the first few hundred epochs the loss falls very slowly (a plateau), and training ends while it is still on that plateau. Also, in Plot 6 one neuron (Neuron 5) is active on about 98% of the samples. ReLU is just $z$ on its active side, so a neuron like that behaves almost like a linear function and adds little curve to the boundary. The experiments below test this explanation.

Write down your prediction before running each experiment:

1. **Learning rate**: as lr goes from 0.005 to 2.0, how does the loss curve change? What happens when lr is too large? (If 2.0 still trains well, add 10 and 20 to `LEARNING_RATES`.)
2. **Hidden width**: what shape is the decision boundary with 1, 2, 5, 10, 20, and 50 hidden units? How many units does it take to separate the two moons?
3. **Training length**: keep lr = 0.05 and change `EPOCHS` to 20,000 in the Section 3 code cell, then run the notebook again from that cell (training takes about 4 times longer). What happens after the plateau?

Change one variable per experiment. Choose settings by looking at the training set only; use test accuracy only to report results.

**Extension (edit the code yourself)**: swap the hidden activation for Tanh. Three things change: $A^{[1]}=\tanh(Z^{[1]})$ in `forward`; the derivative $1-{A^{[1]}}^2$ in `backward`; and Xavier initialization, which in its simple form is $\mathcal N(0, 1/n_{\text{in}})$ (the original Glorot version uses $2/(n_{\text{in}}+n_{\text{out}})$). Copy the functions under new names, e.g. `forward_tanh`, rather than overwriting the versions that already pass the self-checks.

In [ ]:
# Experiment 1: learning-rate comparison (hidden width fixed at N_HIDDEN). Edit LEARNING_RATES and rerun.
LEARNING_RATES = [0.005, 0.05, 0.5, 2.0]

fig, ax = plt.subplots(figsize=(9, 4.5))
for color, lr in zip(SERIES_COLORS, LEARNING_RATES):
    params_lr, hist_lr = train(X_train_std, y_train, n_hidden=N_HIDDEN, learning_rate=lr,
                               epochs=EPOCHS, log_every=LOG_EVERY, seed=SEED, verbose=False)
    tr = accuracy(y_train, predict(X_train_std, params_lr))
    te = accuracy(y_test, predict(X_test_std, params_lr))
    ax.plot(hist_lr["epoch"], hist_lr["loss"], color=color, linewidth=2, label=f"lr = {lr}")
    ax.annotate(f"lr={lr}", xy=(hist_lr["epoch"][-1], hist_lr["loss"][-1]), xytext=(5, 0),
                textcoords="offset points", fontsize=9, color="#52514e", va="center")
    print(f"lr = {lr:<6} final logged loss = {hist_lr['loss'][-1]:.4f}   train acc = {tr:.2%}   test acc = {te:.2%}")
ax.set_yscale("log")
ax.set_xlabel("Epoch")
ax.set_ylabel("Training BCE (log scale)")
ax.set_title(f"Learning-rate comparison (hidden = {N_HIDDEN})")
ax.legend()
fig.tight_layout()
plt.show()

In [ ]:
# Experiment 2: hidden-width comparison. SWEEP_LR defaults to LEARNING_RATE; try 0.5 as well.
HIDDEN_SIZES = [1, 2, 5, 10, 20, 50]
SWEEP_LR = LEARNING_RATE

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
for ax, h in zip(axes.ravel(), HIDDEN_SIZES):
    params_h, _ = train(X_train_std, y_train, n_hidden=h, learning_rate=SWEEP_LR,
                        epochs=EPOCHS, log_every=LOG_EVERY, seed=SEED, verbose=False)
    tr = accuracy(y_train, predict(X_train_std, params_h))
    te = accuracy(y_test, predict(X_test_std, params_h))
    plot_decision_boundary(lambda P, p=params_h: predict_proba(P, p), X_train_std, y_train, ax=ax,
                           title=f"hidden = {h} · train {tr:.1%} · test {te:.1%}")
fig.suptitle(f"Hidden-width comparison (lr = {SWEEP_LR}, epochs = {EPOCHS})", y=1.0)
fig.tight_layout()
plt.show()

## 8. Completion Checklist and Learning Log

- [ ] All 14 functions are implemented and every self-check cell passes.
- [ ] The training set is `(800, 2)`, the test set is `(200, 2)`, and labels are `(m, 1)` column vectors of 0/1.
- [ ] Standardization statistics come from the training set only; the test set reuses the same $\mu$ and $\sigma$.
- [ ] Every gradient has the same shape as its parameter, and the gradient check on real data gives relative errors $<10^{-7}$.
- [ ] `train` records the pre-update loss every 100 epochs and never touches the test set.
- [ ] The loss curve goes down overall, with no NaN or Inf.
- [ ] You plotted the NN vs. Logistic Regression decision boundaries and can explain why their shapes differ.
- [ ] Results stay the same after Restart Kernel → Run All.

### Run log (fill in after you finish)

| Configuration | Train loss | Train acc | Test acc | What you observed |
| --- | --- | --- | --- | --- |
| Default: hidden 5, lr 0.05, 5,000 epochs | | | | |
| Logistic Regression baseline | | | | |
| The best lr from Experiment 1, in your view | | | | |
| The best hidden width from Experiment 2, in your view | | | | |
| Your own combination | | | | |

### Questions to think about (answer in your own words)

1. Why does $dZ^{[2]}=\hat Y-Y$ contain no sigmoid derivative $\hat y(1-\hat y)$?
2. If you removed ReLU ($A^{[1]}=Z^{[1]}$), could the network still learn a curved decision boundary? Why or why not?
3. Why must the weights be initialized randomly, while the biases can start at 0?
4. Why does gradient checking use the centered difference $\frac{J(\theta+\varepsilon)-J(\theta-\varepsilon)}{2\varepsilon}$ instead of the one-sided $\frac{J(\theta+\varepsilon)-J(\theta)}{\varepsilon}$?
5. If a hidden neuron has $Z^{[1]}\le 0$ for every training sample, do its weights still get updated? What is this called?
6. What might cause the loss plateau with the default settings? Which changes got it off the plateau in your experiments?
7. Test accuracy is higher than training accuracy. Is that normal? What could explain it?